# **1. Conectar Google Drive a Colab (Mounti**ng)


Como regla indispensable para trabajar con archivos locales en Colab, preparamos nuestro entorno de trabajo:
* `from google.colab import drive`: Llamamos a la herramienta de conexión.
* `drive.mount('/content/drive')`: Autorizamos el acceso a nuestro almacenamiento en la nube para buscar el archivo CSV.

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **2. Cargar los datos y definir el índice inicial**


Importamos Pandas y abrimos nuestro archivo de las empresas más grandes del mundo:
* `read_csv()`: Como el archivo está separado por comas, usamos esta función.
* `index_col = "Rank"`: Este parámetro especial le indica a Pandas que la columna "Rank" (el puesto en el ranking de la empresa) debe convertirse automáticamente en la etiqueta oficial de las filas, en lugar de usar la numeración por defecto (0, 1, 2...).
* `.head(3)`: Imprimimos las primeras 3 filas para comprobar que el ranking quedó configurado como índice.

In [9]:
import pandas as pd
fortune = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/udmy/aplicada/fortune1000.csv", index_col = "Rank")
fortune.head(3)
fortune.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 1 to 1000
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Company    1000 non-null   object
 1   Sector     1000 non-null   object
 2   Industry   1000 non-null   object
 3   Location   1000 non-null   object
 4   Revenues   1000 non-null   int64 
 5   Profits    1000 non-null   int64 
 6   Employees  1000 non-null   int64 
dtypes: int64(3), object(4)
memory usage: 62.5+ KB


# **3. Agrupar datos por categorías (GroupBy)**


El comando `groupby` es el equivalente en código a las tablas dinámicas de Excel. Nos permite clasificar la información por categorías:
* `groupby('Sector')`: Agrupa todas las empresas de la tabla uniendo aquellas que pertenecen al mismo sector económico (tecnología, energía, salud, etc.). Por sí solo, crea un "objeto de agrupación" que almacena los datos divididos.
* `type(x)`: Nos permite verificar la clase del objeto creado (`DataFrameGroupBy`), confirmando que Pandas ya estructuró la información en bloques independientes por sector.

In [10]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/udmy/aplicada/fortune1000.csv")
sector = df.groupby('Sector')
x = sector
print(type(x))
sector

<class 'pandas.core.groupby.generic.DataFrameGroupBy'>


# **4. Explorar la anatomía de los grupos**


Una vez agrupada la información, podemos consultarla de distintas maneras:
* `sector.size()`: Devuelve una lista con la cantidad exacta de empresas que hay en cada sector económico.
* `sector.first()` / `sector.last()`: Muestran la primera y la última empresa registrada de cada sector.
* `sector.get_group("Retailing")`: Permite "extraer" y aislar únicamente las filas que pertenecen a un sector en específico (por ejemplo, comercio minorista o salud) para analizarlo por separado.

In [11]:
df["Sector"].unique()
sector.size()
sector.first()
sector.last()
sector.get_group("Retailing")
sector.get_group("Health Care")

,Rank,Company,Sector,Industry,Location,Revenues,Profits,Employees
4,5,UnitedHealth Group,Health Care,Health Care: Insurance and Managed Care,"Minnetonka, MN",201159,10558,260000
6,7,CVS Health,Health Care,Health Care: Pharmacy and Other Services,"Woonsocket, RI",184765,6622,203000
24,25,Express Scripts Holding,Health Care,Health Care: Pharmacy and Other Services,"St. Louis, MO",100065,4517,26600
28,29,Anthem,Health Care,Health Care: Insurance and Managed Care,"Indianapolis, IN",90040,3842,56000
36,37,Johnson & Johnson,Health Care,Pharmaceuticals,"New Brunswick, NJ",76450,1300,134000
...,...,...,...,...,...,...,...,...
956,957,AMN Healthcare Services,Health Care,Health Care: Pharmacy and Other Services,"San Diego, CA",1989,132,2980
960,961,IDEXX Laboratories,Health Care,Medical Products and Equipment,"Westbrook, ME",1969,263,7600
994,995,Healthcare Services Group,Health Care,Health Care: Pharmacy and Other Services,"Bensalem, PA",1866,88,55000
996,997,Charles River Laboratories Intl,Health Care,Health Care: Pharmacy and Other Services,"Wilmington, MA",1858,123,11800


# **5. Cálculos masivos y agregaciones múltiples**


Podemos aplicar operaciones matemáticas a nuestros grupos para obtener resúmenes financieros:
* `sector["Employees"].sum()` / `.mean()`: Suma o promedia la cantidad de empleados por cada sector económico de forma automática.
* `.agg({...})`: Es una herramienta súper potente que nos permite aplicar varias operaciones a diferentes columnas al mismo tiempo (por ejemplo, calcular el promedio y el máximo de los ingresos, y la suma de empleados en una sola línea de código).

In [12]:
sector["Employees"].sum()
sector["Employees"].mean()
sector.agg({"Revenues" : "mean", "Profits" : "mean", "Employees" : "sum"})
sector.agg({"Revenues" : ["mean", "max"], "Profits" : ["max", "min"], "Employees" : "sum"})

Revenues         Profits       Employees
                                        mean     max     max   min       sum
Sector                                                                      
Aerospace &  Defense            15353.400000   93392    8197   -74   1010124
Apparel                          7225.500000   34350    4240  -479    355699
Business Services                5963.962264   21034    6699  -558   1593999
Chemicals                        7610.636364   62683    3000  -297    474020
Energy                          14425.299065  244363   19710 -5723    981207
Engineering &  Construction      6399.333333   19521    1038  -333    420745
Financials                      15757.935484  242137   44940 -6798   3500119
Food &  Drug Stores             33789.000000  122662    4078  -374   1398074
Food, Beverages &  Tobacco      13790.054054   63525   10999  -287   1079316
Health Care                     21239.309859  201159   21308 -2459   2971189
Hotels, Restaurants &  Leisure   6916.346154   22894    5192  -375   2304337
Household Products               8277.821429   66217   15326 -1054    674896
Industrials                     10615.102041  122274    4858 -5786   1566110
Materials                        6184.400000   23302    2144  -258    653751
Media                            9219.480000   55137    8980  -738    545985
Motor Vehicles &  Parts         22817.631579  157311    7602 -3864    943582
Retailing                       21874.714286  500343    9862 -1068   6800658
Technology                      13347.786408  229234   48351 -3728   3196807
Telecommunications              46695.900000  160546   30101 -2117    785609
Transportation                  11379.000000   65872   10712  -107   1652113
Wholesalers                     20185.204545  198533    5070  -267    744175

# **6. Recorrer grupos y extraer máximos usando bucles**


Podemos usar un bucle de Python (`for`) para entrar a cada sector de forma automática y extraer datos específicos:
* `nlargest(1, "Profits")`: Es una función que busca la fila con el valor numérico más alto (en este caso, la empresa con mayor ganancia de cada sector).
* El bucle recorre cada sector, saca su empresa más rentable y las va acumulando dentro de una nueva tabla vacía (`pd.DataFrame`) para generar un reporte personalizado.

In [13]:
sectors = fortune.groupby('Sector')
df_empty = pd.DataFrame(columns = fortune.columns)

for sector_name, values in sectors:
  company_with_highest_profit = values.nlargest(1, "Profits")
  df_empty = pd.concat([df_empty, company_with_highest_profit])

df_empty

,Company,Sector,Industry,Location,Revenues,Profits,Employees
27,Boeing,Aerospace & Defense,Aerospace and Defense,"Chicago, IL",93392,8197,140800
89,Nike,Apparel,Apparel,"Beaverton, OR",34350,4240,74400
161,Visa,Business Services,Financial Data Services,"SF, CA",18358,6699,15000
345,Air Products & Chemicals,Chemicals,Chemicals,"Allentown, PA",8442,3000,15150
2,Exxon Mobil,Energy,Petroleum Refining,"Irving, TX",244363,19710,71200
211,D.R. Horton,Engineering & Construction,Homebuilders,"Arlington, TX",14091,1038,7735
3,Berkshire Hathaway,Financials,Insurance: Property and Casualty (Stock),"Omaha, NE",242137,44940,377000
19,Walgreens Boots Alliance,Food & Drug Stores,Food and Drug Stores,"Deerfield, IL",118214,4078,290000
114,Kraft Heinz,"Food, Beverages & Tobacco",Food Consumer Products,"Pittsburgh, PA",26232,10999,39000
57,Pfizer,Health Care,Pharmaceuticals,"New York, NY",52546,21308,90200


# **7. Resumen del proceso completo**


En conclusión, este archivo nos enseña el potencial de la estadística descriptiva y el resumen de datos masivos:
* **Estructuración por categorías:** Aprendimos a utilizar `.groupby()` para sectorizar bases de datos complejas, imitando la lógica de las tablas dinámicas.
* **Agregaciones avanzadas:** Dominamos el método `.agg()` para realizar múltiples operaciones estadísticas simultáneas sobre distintas columnas (medias, máximos, mínimos y sumas).
* **Automatización:** Comprendimos cómo combinar los grupos con bucles de programación para filtrar y extraer información clave (como el récord de ganancias por sector).